# PROJECT URAAN
## Pakistan Grocery Economic Analysis

Research focus:
Inflation, commodity prices and grocery
retail financial resilience.

Country: Pakistan
Currency: PKR
Primary environment: Google Colab

Data policy:
Use real publicly available datasets.
Document all sources and limitations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

print("PROJECT URAAN")
print("Research environment initialised.")

In [ ]:
url = (
    "https://huggingface.co/datasets/"
    "opendoc-pakistan/pakistan_inflation_and_prices/"
    "resolve/main/data/processed/"
    "monthly_spi_city_item_prices.csv"
)

df_prices = pd.read_csv(url)

print("Dataset successfully loaded!")
print("Shape:", df_prices.shape)

display(df_prices.head())

In [ ]:
print("COLUMN NAMES")
print(df_prices.columns.tolist())

print("\nDATA TYPES")
print(df_prices.dtypes)

print("\nDATASET INFORMATION")
df_prices.info()

print("\nMISSING VALUES")
display(df_prices.isnull().sum())

print("\nDESCRIPTIVE STATISTICS")
display(df_prices.describe(include="all"))

##  — DATA QUALITY AUDIT

In [ ]:
#  Checking date coverage
print("DATE COVERAGE")
print("Earliest period:", df_prices["period"].min())
print("Latest period:", df_prices["period"].max())
print("Unique periods:", df_prices["period"].nunique())


#  Investigating item sets
print("\nITEM SET DISTRIBUTION")
display(df_prices["item_set"].value_counts())


#  Investigating availability statuses
print("\nAVAILABILITY STATUS")
display(df_prices["availability_status"].value_counts())


#  Investigating negative prices
negative_prices = df_prices[
    df_prices["average_price_pkr"] < 0
]

print("\nNEGATIVE PRICE RECORDS")
print("Count:", len(negative_prices))
display(negative_prices)


#  Investigating missing prices
missing_prices = df_prices[
    df_prices["average_price_pkr"].isna()
]

print("\nMISSING PRICE RECORDS")
print("Count:", len(missing_prices))

display(
    missing_prices.groupby(
        ["item_set", "availability_status"]
    ).size().reset_index(name="count")
)


# Checking duplicate records
print("\nDUPLICATE CHECK")

print(
    "Exact duplicate rows:",
    df_prices.duplicated().sum()
)

key_columns = [
    "period",
    "item_set",
    "item_number",
    "city"
]

print(
    "Duplicate business keys:",
    df_prices.duplicated(subset=key_columns).sum()
)


#  Investigating units
print("\nUNIT DISTRIBUTION")
display(df_prices["unit"].value_counts())


#  Investigating commodity coverage
print("\nTOP 20 COMMODITIES")
display(
    df_prices["item_description"]
    .value_counts()
    .head(20)
)

##  — SOURCE STRUCTURE AUDIT

In [ ]:
#  Inspecting all distinct city values
print("UNIQUE CITY VALUES")

display(
    df_prices["city"]
    .value_counts()
    .to_frame("count")
)


#  Finding suspicious city labels
suspicious_city = df_prices[
    df_prices["city"].str.contains(
        "%change|change",
        case=False,
        na=False
    )
]

print("\nSUSPICIOUS CITY RECORDS")
print("Count:", len(suspicious_city))

display(suspicious_city.head(15))


#  Inspecting reported-zero records
zero_records = df_prices[
    df_prices["availability_status"] == "reported_zero"
]

print("\nREPORTED ZERO RECORDS")
print("Count:", len(zero_records))

display(zero_records.head(20))


#  Checking actual numerical values in zero records
print("\nREPORTED ZERO PRICE VALUES")

display(
    zero_records["average_price_pkr"]
    .value_counts(dropna=False)
)


#  Inspecting missing records by item set
print("\nMISSING PRICE RATE BY ITEM SET")

missing_summary = (
    df_prices.assign(
        missing_price=df_prices["average_price_pkr"].isna()
    )
    .groupby("item_set")
    .agg(
        total_records=("average_price_pkr", "size"),
        missing_records=("missing_price", "sum")
    )
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_records"]
    / missing_summary["total_records"] * 100
)

display(missing_summary)


#  Inspecting item sets and their units
print("\nITEM SET AND UNIT COMBINATIONS")

display(
    df_prices.groupby(
        ["item_set", "unit"]
    ).size().reset_index(name="records")
)

In [ ]:
# Identifying records with actual city names
valid_cities = [
    "Islamabad", "Rawalpindi", "Gujranwala", "Sialkot",
    "Lahore", "Faisalabad", "Sargodha", "Multan",
    "Bahawalpur", "Karachi", "Hyderabad"
]

print("All distinct city labels:")
print(sorted(df_prices["city"].dropna().unique()))

print("\nRecords matching the known city list:")
print(df_prices["city"].isin(valid_cities).sum())

print("\nRecords outside the known city list:")
print((~df_prices["city"].isin(valid_cities)).sum())

In [ ]:
actual_cities = [
    "Bahawalpur", "Bannu", "Faisalabad", "Gujranwala",
    "Hyderabad", "Islamabad", "Karachi", "Khuzdar",
    "Lahore", "Larkana", "Multan", "Peshawar",
    "Quetta", "Rawalpindi", "Sargodha", "Sialkot",
    "Sukkur"
]

city_records = df_prices[
    df_prices["city"].isin(actual_cities)
]

print("Number of actual cities:", len(actual_cities))
print("City-level records:", len(city_records))

print("\nRecords per city:")
print(city_records["city"].value_counts().sort_index())

print("\nRemaining non-city labels:")
print(
    df_prices.loc[
        ~df_prices["city"].isin(actual_cities),
        "city"
    ].value_counts().to_string()
)

In [ ]:
# Counting each record type
city_mask = df_prices["city"].isin(actual_cities)

average_mask = df_prices["city"].str.startswith(
    "Average", na=False
)

change_mask = df_prices["city"].str.startswith(
    "%change", na=False
)

print("Total dataset rows:", len(df_prices))
print("City records:", city_mask.sum())
print("Average records:", average_mask.sum())
print("Percentage-change records:", change_mask.sum())

print("\nDo all categories account for the dataset?")
print(
    city_mask.sum()
    + average_mask.sum()
    + change_mask.sum()
    == len(df_prices)
)

In [ ]:
city_data = df_prices[
    df_prices["city"].isin(actual_cities)
]

print("City-level dataset shape:", city_data.shape)

print("\nMissing prices:")
print(city_data["average_price_pkr"].isna().sum())

print("\nNegative prices:")
print((city_data["average_price_pkr"] < 0).sum())

print("\nAvailability status:")
print(city_data["availability_status"].value_counts())

print("\nPrice summary:")
display(city_data["average_price_pkr"].describe())

In [ ]:
missing_by_city = (
    city_data.assign(
        price_missing=city_data["average_price_pkr"].isna()
    )
    .groupby("city")
    .agg(
        total_records=("price_missing", "size"),
        missing_prices=("price_missing", "sum")
    )
)

missing_by_city["missing_percentage"] = (
    missing_by_city["missing_prices"]
    / missing_by_city["total_records"] * 100
)

display(missing_by_city.round(2))

In [ ]:
missing_by_item = (
    city_data.assign(
        price_missing=city_data["average_price_pkr"].isna()
    )
    .groupby(["item_description", "unit"])
    .agg(
        total_records=("price_missing", "size"),
        missing_prices=("price_missing", "sum")
    )
)

missing_by_item = missing_by_item[
    missing_by_item["missing_prices"] > 0
].sort_values("missing_prices", ascending=False)

print("Products with missing prices:", len(missing_by_item))

display(missing_by_item.head(15))

# The product coverage

In [ ]:
print("Distinct products by item set:")

display(
    city_data.groupby("item_set")["item_description"]
    .nunique()
    .to_frame("unique_products")
)

print("\nProducts in the dataset:")

display(
    city_data[["item_description", "unit", "item_set"]]
    .drop_duplicates()
    .sort_values(["item_set", "item_description"])
    .reset_index(drop=True)
)

# Inspecting only the SPI product names

In [ ]:
spi_products = (
    city_data.loc[
        city_data["item_set"] == "spi_index_items",
        ["item_description", "unit"]
    ]
    .drop_duplicates()
    .sort_values("item_description")
)

print("SPI products:", len(spi_products))
print(spi_products.to_string(index=False))

# Classify the SPI products

In [ ]:
food_products = [
    "Bananas (Kela) Local",
    "Beef with Bone (Average Quality)",
    "Bread plain (Small Size)",
    "Chicken Farm Broiler (Live)",
    "Chilies Powder NATIONAL 200 gm Packet",
    "Cooked Beef at Average Hotel",
    "Cooked Daal at Average Hotel",
    "Cooking Oil DALDA or Other Similar Brand (SN), 5 Litre Tin",
    "Curd (Dahi) Loose",
    "Eggs Hen (Farm)",
    "Garlic (Lehsun)",
    "Gur (Average Quality)",
    "Milk fresh (Un-boiled)",
    "Mustard Oil (Average Quality)",
    "Mutton (Average Quality)",
    "Onions",
    "Potatoes",
    "Powdered Milk NIDO 390 gm Polybag",
    "Pulse Gram",
    "Pulse Mash (Washed)",
    "Pulse Masoor (Washed)",
    "Pulse Moong (Washed)",
    "Rice Basmati Broken (Average Quality)",
    "Rice IRRI-6/9 (Sindh/Punjab)",
    "Salt Powdered (NATIONAL/SHAN) 800 gm Packet",
    "Sugar Refined",
    "Tea Lipton Yellow Label 190 gm Packet",
    "Tea Prepared Ordinary",
    "Tomatoes",
    "Vegetable Ghee DALDA/HABIB 2.5 kg Tin",
    "Vegetable Ghee DALDA/HABIB or Other superior Quality 1 kg Pouch",
    "Wheat Flour Bag"
]

household_products = [
    "Energy Saver Philips 14 Watt",
    "Firewood Whole",
    "Match Box",
    "Sufi Washing Soap 250 gm Cake",
    "Toilet Soap LIFEBUOY 115 gm"
]

energy_products = [
    "Electricity Charges upto 50 Units",
    "Gas Charges upto 3.3719 MMBTU",
    "Hi-Speed Diesel",
    "LPG 11.67 kg Cylinder",
    "Petrol Super"
]

spi_products = spi_products.copy()

def classify_product(name):
    if name in food_products:
        return "Food and groceries"
    elif name in household_products:
        return "Household essentials"
    elif name in energy_products:
        return "Energy and utilities"
    else:
        return "Other goods and services"

spi_products["category"] = spi_products["item_description"].apply(
    classify_product
)

display(
    spi_products.groupby("category")
    .size()
    .to_frame("product_count")
)

display(spi_products)

# Checking monthly coverage of grocery prices

In [ ]:
food_names = spi_products.loc[
    spi_products["category"] == "Food and groceries",
    "item_description"
].tolist()

food_data = city_data[
    (city_data["item_set"] == "spi_index_items") &
    (city_data["item_description"].isin(food_names))
].copy()

print("Food records:", len(food_data))
print("Unique food products:", food_data["item_description"].nunique())
print("Cities:", food_data["city"].nunique())
print("Months:", food_data["period"].nunique())

print("\nRecords per month:")
print(food_data["period"].value_counts().sort_index())

In [ ]:
spi_city_data = city_data[
    city_data["item_set"] == "spi_index_items"
]

print("SPI records by month:")
print(
    spi_city_data.groupby("period").size()
)

print("\nAugust 2025 records:")
print(
    spi_city_data[
        spi_city_data["period"] == "2025-08"
    ].shape[0]
)

print("\nFood prices missing:")
print(food_data["average_price_pkr"].isna().sum())

In [ ]:
food_missing = (
    food_data.groupby("item_description")
    .agg(
        total_records=("average_price_pkr", "size"),
        missing_prices=("average_price_pkr",
                        lambda x: x.isna().sum())
    )
)

food_missing["missing_percentage"] = (
    food_missing["missing_prices"]
    / food_missing["total_records"] * 100
)

food_missing = food_missing[
    food_missing["missing_prices"] > 0
].sort_values("missing_prices", ascending=False)

print("Food products with missing prices:", len(food_missing))
display(food_missing)

In [ ]:
food_coverage = (
    food_data.groupby("period")
    .agg(
        total_records=("item_description", "size"),
        valid_prices=("average_price_pkr", "count"),
        missing_prices=("average_price_pkr",
                        lambda x: x.isna().sum())
    )
)

display(food_coverage)

print("\nFood observations by city:")
print(food_data.groupby("city").size().to_string())

# Locating missing rice observations

In [ ]:
rice_missing = food_data[
    (food_data["item_description"] ==
     "Rice IRRI-6/9 (Sindh/Punjab)") &
    (food_data["average_price_pkr"].isna())
]

print("Total missing rice observations:", len(rice_missing))

print("\nMissing observations by city:")
display(
    rice_missing.groupby("city")
    .size()
    .to_frame("missing_months")
    .sort_values("missing_months", ascending=False)
)

In [ ]:
rice_data = food_data[
    food_data["item_description"] ==
    "Rice IRRI-6/9 (Sindh/Punjab)"
]

rice_coverage = (
    rice_data.groupby("city")
    .agg(
        total_months=("period", "nunique"),
        valid_prices=("average_price_pkr", "count")
    )
)

display(rice_coverage)

print("\nCities with complete coverage:",
      (rice_coverage["valid_prices"] == 11).sum())

print("Cities with no reported prices:",
      (rice_coverage["valid_prices"] == 0).sum())

## checking missing prices across all food products

In [ ]:
product_coverage = (
    food_data.groupby("item_description")
    .agg(
        total_records=("average_price_pkr", "size"),
        valid_prices=("average_price_pkr", "count")
    )
)

product_coverage["missing_prices"] = (
    product_coverage["total_records"] -
    product_coverage["valid_prices"]
)

display(
    product_coverage[
        product_coverage["missing_prices"] > 0
    ]
)

In [ ]:
price_ranges = (
    food_data.groupby(["item_description", "unit"])
    ["average_price_pkr"]
    .agg(["count", "min", "median", "max"])
    .round(2)
    .sort_values("median", ascending=False)
)

display(price_ranges)

In [ ]:
flour_data = food_data[
    food_data["item_description"] == "Wheat Flour Bag"
]

flour_trend = (
    flour_data.groupby("period")["average_price_pkr"]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)

display(flour_trend)

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    flour_trend.index,
    flour_trend["mean"],
    marker="o",
    linewidth=2,
    label="Average price"
)

plt.fill_between(
    flour_trend.index,
    flour_trend["min"],
    flour_trend["max"],
    alpha=0.15,
    label="Minimum–maximum city prices"
)

plt.title("Wheat Flour (20 kg): Monthly Price Trend")
plt.xlabel("Month")
plt.ylabel("Price (PKR)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
flour_trend["city_price_spread"] = (
    flour_trend["max"] - flour_trend["min"]
)

flour_trend["spread_pct_of_mean"] = (
    flour_trend["city_price_spread"] /
    flour_trend["mean"] * 100
).round(2)

display(
    flour_trend[
        ["mean", "min", "max",
         "city_price_spread", "spread_pct_of_mean"]
    ]
)

In [ ]:
plt.figure(figsize=(12, 5))

sns.boxplot(
    data=flour_data,
    x="period",
    y="average_price_pkr"
)

plt.title("Wheat Flour Prices Across 17 Pakistani Cities")
plt.xlabel("Month")
plt.ylabel("Price (PKR per 20 kg bag)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Calculating month-to-month price changes by city

In [ ]:
flour_city_changes = (
    flour_data.pivot(
        index="city",
        columns="period",
        values="average_price_pkr"
    )
)

flour_city_changes = flour_city_changes.sort_index(axis=1)

flour_city_changes_pct = (
    flour_city_changes.pct_change(axis=1) * 100
).round(2)

display(flour_city_changes_pct)

## quantifying the city-wide pattern

In [ ]:
change_summary = pd.DataFrame({
    "median_change_pct": flour_city_changes_pct.median(),
    "cities_increased": (flour_city_changes_pct > 0).sum(),
    "cities_decreased": (flour_city_changes_pct < 0).sum(),
    "cities_unchanged": (flour_city_changes_pct == 0).sum()
})

display(change_summary.round(2))

In [ ]:
city_cumulative_change = pd.DataFrame({
    "july_price": flour_city_changes["2025-07"],
    "june_price": flour_city_changes["2026-06"]
})

city_cumulative_change["total_change_pct"] = (
    (
        city_cumulative_change["june_price"] /
        city_cumulative_change["july_price"]
    ) - 1
) * 100

display(
    city_cumulative_change
    .sort_values("total_change_pct", ascending=False)
    .round(2)
)

## verifying the wheat flour source records

In [ ]:
flour_sources = (
    flour_data.groupby("source_url")
    .agg(
        records=("average_price_pkr", "size"),
        first_period=("period", "min"),
        last_period=("period", "max")
    )
)

display(flour_sources)

print("\nProduct identifiers:")
display(
    flour_data[
        ["item_number", "item_description", "unit"]
    ].drop_duplicates()
)

In [ ]:
flour_check = flour_data[
    flour_data["period"].isin(
        ["2025-07", "2025-09", "2025-10"]
    )
][
    ["period", "city", "average_price_pkr", "source_url"]
].sort_values(["city", "period"])

display(flour_check)

In [ ]:
pbs_files = {
    "July 2025": flour_data.loc[
        flour_data["period"] == "2025-07",
        "source_url"
    ].iloc[0],

    "September 2025": flour_data.loc[
        flour_data["period"] == "2025-09",
        "source_url"
    ].iloc[0]
}

for month, url in pbs_files.items():
    workbook = pd.ExcelFile(url)
    print(f"\n{month}")
    print("Sheets:", workbook.sheet_names)

In [ ]:
for month, url in pbs_files.items():
    original = pd.read_excel(
        url,
        sheet_name="Items 1-51"
    )

    print(f"\n{month}")
    print("Shape:", original.shape)
    print("Columns:", original.columns.tolist())
    display(original.head(5))

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51"
    )

    row = original[
        original.iloc[:, 1] == "Wheat Flour Bag"
    ]

    print(f"\n{month}")
    print(row.iloc[:, 20:25].to_string(index=False))

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51"
    )

    print(f"\n{month}")
    print(original.iloc[0, 20:25].to_string())

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51",
        header=None
    )

    print(f"\n{month}")
    print(original.iloc[0, 20:25].tolist())

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51",
        header=None
    )

    print(f"\n{month}")
    print(original.iloc[:5, 20:25].to_string(index=True, header=False))

## comparing calculated city average with PBS

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51",
        header=None
    )

    row = original[
        original.iloc[:, 1] == "Wheat Flour Bag"
    ]

    city_prices = pd.to_numeric(
        row.iloc[0, 3:20],
        errors="coerce"
    )

    print(f"\n{month}")
    print("PBS city mean:", round(city_prices.mean(), 2))
    print("PBS reported average:", row.iloc[0, 20])

In [ ]:
for month, url in pbs_files.items():

    summary = pd.read_excel(
        url,
        sheet_name="Sheet1",
        header=None
    )

    print(f"\n{month}")
    print("Shape:", summary.shape)
    print(summary.head(12).to_string(index=False))

In [ ]:
for month, url in pbs_files.items():

    extra = pd.read_excel(
        url,
        sheet_name="Items 52-66",
        header=None
    )

    print(f"\n{month}")
    print("Shape:", extra.shape)
    print(extra.head(5).to_string(index=False))

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51",
        header=None
    )

    rows = original.iloc[2:53].copy()

    results = []

    for _, row in rows.iterrows():

        city_prices = pd.to_numeric(
            row.iloc[3:20],
            errors="coerce"
        )

        pbs_average = pd.to_numeric(
            row.iloc[20],
            errors="coerce"
        )

        results.append({
            "item": row.iloc[1],
            "city_mean": city_prices.mean(),
            "pbs_average": pbs_average,
            "difference": city_prices.mean() - pbs_average
        })

    comparison = pd.DataFrame(results)

    print(f"\n{month}")
    print(comparison.round(2).to_string(index=False))

In [ ]:
for month, url in pbs_files.items():

    original = pd.read_excel(
        url,
        sheet_name="Items 1-51",
        header=None
    )

    rows = original.iloc[2:53].copy()

    city_means = rows.iloc[:, 3:20].apply(
        pd.to_numeric,
        errors="coerce"
    ).mean(axis=1)

    pbs_means = pd.to_numeric(
        rows.iloc[:, 20],
        errors="coerce"
    )

    differences = (city_means - pbs_means).abs()

    print(f"\n{month}")
    print("Total items:", len(rows))
    print("Matching within PKR 0.01:", (differences <= 0.01).sum())
    print("Different by more than PKR 0.01:", (differences > 0.01).sum())
    print("Largest absolute difference:", round(differences.max(), 2))
    print("Mean absolute difference:", round(differences.mean(), 2))

## investigating whether the differences are systematic

In [ ]:
september_check = comparison.copy()

september_check["abs_difference"] = (
    september_check["difference"].abs()
)

september_check.nlargest(
    5, "abs_difference"
)[
    ["item", "city_mean", "pbs_average", "difference"]
].round(2)

In [ ]:
firewood = original[
    original.iloc[:, 1] == "Firewood Whole"
]

firewood_values = pd.DataFrame({
    "city": original.iloc[1, 3:20].values,
    "price": pd.to_numeric(
        firewood.iloc[0, 3:20],
        errors="coerce"
    ).values
})

print(firewood_values.to_string(index=False))

In [ ]:
print("Number of observations:", len(firewood_values))

print(
    "Calculated sum:",
    firewood_values["price"].sum()
)

print(
    "Calculated mean:",
    firewood_values["price"].mean()
)

print(
    "PBS reported average:",
    pd.to_numeric(firewood.iloc[0, 20])
)

### Source Validation: PBS Monthly SPI Average Prices

A comparison of the arithmetic mean of 17 displayed city prices
with the PBS-reported average revealed discrepancies across
multiple SPI items.

For September 2025, 41 of 51 items differed by more than PKR 0.01.
The mean absolute difference was PKR 3.40.

Official PBS documentation confirms that SPI prices are collected
across multiple markets in 17 cities. However, the exact aggregation
procedure used for the monthly published average-price column
has not yet been independently established.

Therefore, this study retains two separate measures:

1. PBS-reported average price.
2. Independently calculated 17-city arithmetic mean.

Neither measure is substituted for the other.
Further methodological verification is required.

In [ ]:
market_counts = {
    "Islam-abad": 4,
    "Rawal-pindi": 6,
    "Gujran-wala": 1,
    "Sialkot": 1,
    "Lahore": 7,
    "Faisal-abad": 2,
    "Sar-godha": 1,
    "Multan": 3,
    "Baha-walpur": 1,
    "Karachi": 13,
    "Hyder-abad": 4,
    "Sukkur": 2,
    "Larkana": 1,
    "Pesha-war": 3,
    "Bannu": 1,
    "Quetta": 2,
    "Khuz-dar": 1
}

firewood_values["market_count"] = (
    firewood_values["city"].map(market_counts)
)

weighted_mean = np.average(
    firewood_values["price"],
    weights=firewood_values["market_count"]
)

print("Historical market-weighted mean:", round(weighted_mean, 2))
print("PBS reported average:", 1351.16)